# PhishGuard-PK | Step 50: Supervised URL AI, external research gate

**One click:** Open in Google Colab, select **Runtime → Run all**. CPU is enough. No dataset upload and no code edits. The 10,000 official UCI PhiUSIIL (2024) source-labelled URL samples are embedded in this notebook. It attempts to download only the two fixed publisher files: OpenPhish community feed and a dated Tranco ranking. **It NEVER opens a phishing URL.**

Workflow: UCI 2024 label audit (original 1 legitimate, 0 phishing) → domain-disjoint 70/15/15 split → train character TF-IDF and lexical Random Forest → model/threshold selection on *validation only* → held-out UCI test → fresh OpenPhish provider-labelled URLs and Tranco homepage proxy evaluation. It removes overlapping registered domains between external cohort and historical training.

**Security:** Output file always labels the model **RESEARCH_ONLY**; never upload it into the live website. Popular Tranco domains are not verified legitimate pages, and OpenPhish feed records are not individually verified Pakistani phishing cases. The target is independent evidence, not an inflated accuracy claim.

If publisher files cannot be downloaded, the output will clearly state **SOURCE_UNAVAILABLE**, while preserving the UCI-only experimental evaluation. Send the resulting ZIP to this chat; no GPU needed.


In [ ]:
from pathlib import Path
import base64, gzip, json, zipfile
ROOT=Path('/content/phishguard_step50')
for path in ('phishguard','scripts','input','results'):
 (ROOT/path).mkdir(parents=True,exist_ok=True)
(ROOT/'phishguard'/'__init__.py').write_text('',encoding='utf-8')
(ROOT/'scripts'/'__init__.py').write_text('',encoding='utf-8')
(ROOT/'phishguard'/'url_policy.py').write_text('"""Step 44: conservative, offline URL-structure assessment.\n\nNever fetches links, resolves DNS, expands shorteners, or authenticates a domain.\nThe indicator is not a probability of phishing. Reference domains exist ONLY\nfor lookalike detection, never for automatically declaring a URL safe.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom urllib.parse import urlsplit\nimport ipaddress\nimport re\n\n# A small, dated reference set, NOT a complete official registry / allowlist.\n# All reference domain comparisons require a dot boundary.\nREFERENCE_DOMAINS = {\n    \'github\': (\'github.com\', \'github.io\'),\n    \'wikipedia\': (\'wikipedia.org\', \'wikimedia.org\'),\n    \'python\': (\'python.org\', \'pypi.org\'),\n    \'google\': (\'google.com\', \'google.com.pk\', \'googleapis.com\', \'googleusercontent.com\'),\n    \'microsoft\': (\'microsoft.com\', \'microsoftonline.com\'),\n    \'bisp\': (\'bisp.gov.pk\',),\n    \'jazzcash\': (\'jazzcash.com.pk\',),\n    \'meezan\': (\'meezanbank.com\', \'meezanbank.pk\'),\n    \'fbr\': (\'fbr.gov.pk\',),\n    \'hbl\': (\'hbl.com\',),\n    \'alfalah\': (\'bankalfalah.com\',),\n    \'easypaisa\': (\'easypaisa.com.pk\',),\n}\n# A namespace marker, not an assertion that every *.gov.pk site is authentic.\n# Used solely to detect gov.pk appended to an unrelated destination hostname.\nPUBLIC_SECTOR_NAMESPACE_MARKERS = (\'gov.pk\',)\nSHORTENERS = frozenset({\'bit.ly\', \'tinyurl.com\', \'t.co\', \'cutt.ly\', \'is.gd\', \'rb.gy\', \'ow.ly\', \'shorturl.at\'})\nAUTH_ACTION = re.compile(r\'(?i)(?:^|[/?#&=_-])(?:login|signin|sign-in|verify|verification|reset-password|password|otp|kyc|claim-prize|claim)(?:$|[/?#&=_-])\')\nSCAM_BAIT = re.compile(r\'(?i)(?:^|[/?#&=_-])(?:prize|winner|free-money|claim-reward|cash-reward|bisp-payment|8171-payment)(?:$|[/?#&=_-])\')\n\n\ndef _is_domain_or_subdomain(host: str, domain: str) -> bool:\n    return host == domain or host.endswith(\'.\' + domain)\n\n\ndef _brand_host_signal(host: str):\n    """(impersonation severity, reason) based on embedded brand/domain markers.\n\n    No ownership claim: official domains may be compromised and lookalike sites\n    may be benign. Signals are intentionally conservative.\n    """\n    for marker in PUBLIC_SECTOR_NAMESPACE_MARKERS:\n        if _is_domain_or_subdomain(host, marker):\n            continue\n        if host.startswith(marker + \'.\') or (\'.\' + marker + \'.\') in host or host.startswith(marker + \'-\'):\n            return 88, \'Hostname includes a government-style .gov.pk marker but points to another domain.\'\n        dashed = marker.replace(\'.\', \'-\')\n        if (host.startswith(dashed + \'.\') or host.startswith(dashed + \'-\')) and re.search(\n            r\'(?i)(?:verify|payment|8171|challan|account|login|claim)\', host\n        ):\n            return 82, \'Hostname imitates a government domain spelling with payment or verification language.\'\n    for brand, official_domains in REFERENCE_DOMAINS.items():\n        if any(_is_domain_or_subdomain(host, domain) for domain in official_domains):\n            continue\n        # An entire reference domain prepended to a different hostname is a\n        # strong deceptive-host signal: github.com.evil.org != github.com.\n        if any(host.startswith(domain + \'.\') or (\'.\' + domain + \'.\') in host\n               or host.startswith(domain + \'-\') for domain in official_domains):\n            return 88, f\'Domain resembles {brand} but the actual hostname is different; check the full hostname.\'\n        # Entire official domain with dots swapped for hyphens, for example\n        # bisp-gov-pk.verify.example. Strong spoof cue only when combined with\n        # account/payment context; do not classify all mentions as phishing.\n        for domain in official_domains:\n            dashed = domain.replace(\'.\', \'-\')\n            if (host.startswith(dashed + \'.\') or host.startswith(dashed + \'-\')) and re.search(\n                r\'(?i)(?:login|verify|payment|8171|claim|otp|kyc|prize|reward|account)\', host\n            ):\n                return 82, f\'Hostname uses a hyphenated spelling of a {brand} reference with a sensitive action.\'\n        # Bounded brand tokens with login / verification cues can be suspicious;\n        # ordinary brand-related fan/project domains are not automatically scams.\n        if re.search(r\'(?i)(?:^|[.-])\' + re.escape(brand) + r\'(?:[.-]|$)\', host):\n            if re.search(r\'(?i)(?:login|verify|secure|claim|payment|support|kyc|account|prize|8171|reward|bonus)\', host):\n                return 68, f\'Hostname combines a {brand} reference with account or verification language.\'\n            return 33, f\'Hostname mentions {brand}, but ownership is not verified.\'\n    return 0, \'\'\n\n\n@dataclass(frozen=True)\nclass URLAssessment:\n    indicator: int\n    reasons: tuple[str, ...]\n    hostname: str\n    status: str\n    high_confidence_structure: bool\n\n\ndef assess_url(url: str) -> URLAssessment:\n    """Returns interpretable offline indicator 0-100; NOT a fraud probability."""\n    if not isinstance(url, str) or not url.strip() or len(url) > 10000:\n        return URLAssessment(70, (\'URL could not be parsed safely.\',), \'\', \'invalid\', True)\n    raw = url.strip()\n    candidate = raw if re.match(r\'(?i)^https?://\', raw) else \'http://\' + raw\n    try:\n        parts = urlsplit(candidate)\n        host = (parts.hostname or \'\').strip(\'.\').lower()\n        port = parts.port  # validates invalid port values\n        if not host or any(c.isspace() for c in host) or \'\\\\\' in raw or \'%\' in host:\n            raise ValueError(\'Invalid hostname\')\n        # Convert unicode domains for reliable ASCII label processing.\n        ascii_host = host.encode(\'idna\').decode(\'ascii\')\n        if len(ascii_host) > 253 or any(len(label) > 63 or not label for label in ascii_host.split(\'.\')):\n            raise ValueError(\'Invalid DNS labels\')\n    except (ValueError, UnicodeError):\n        return URLAssessment(70, (\'Malformed or ambiguous URL structure. Avoid opening it.\',), \'\', \'invalid\', True)\n\n    score = 8\n    reasons = []\n    high = False\n    brand_score, brand_reason = _brand_host_signal(ascii_host)\n    if brand_score:\n        score = max(score, brand_score)\n        reasons.append(brand_reason)\n        high |= brand_score >= 75\n\n    if parts.username is not None or parts.password is not None or \'@\' in parts.netloc:\n        score = max(score, 82)\n        reasons.append(\'URL contains @ / userinfo: the apparent domain before @ may not be the destination.\')\n        high = True\n\n    ip = None\n    try:\n        ip = ipaddress.ip_address(ascii_host.strip(\'[]\'))\n    except ValueError:\n        pass\n    if ip:\n        if ip.is_global:\n            score = max(score, 45)\n            reasons.append(\'Destination uses a numeric IP rather than a domain name.\')\n        else:\n            score = max(score, 28)\n            reasons.append(\'Destination is an internal, private or reserved IP address; cannot verify it as a public site.\')\n\n    if ascii_host in SHORTENERS:\n        score = max(score, 48)\n        reasons.append(\'Shortened link hides its final destination; it has not been expanded.\')\n\n    if \'xn--\' in ascii_host or any(ord(c) > 127 for c in host):\n        score = max(score, 48)\n        reasons.append(\'Internationalized / punycode hostname; visually similar letters may be confusing.\')\n\n    if len(ascii_host.split(\'.\')) >= 5:\n        score = max(score, 31)\n        reasons.append(\'Hostname has several subdomain levels; inspect its real ending carefully.\')\n\n    # Only a compounded, contextual signal. HTTP alone is not phishing;\n    # HTTPS alone is not proof of legitimacy.\n    pathquery = (parts.path or \'\') + \'?\' + (parts.query or \'\')\n    action = bool(AUTH_ACTION.search(pathquery))\n    bait = bool(SCAM_BAIT.search(pathquery))\n    if action and parts.scheme.lower() == \'http\':\n        score = max(score, 52)\n        reasons.append(\'Account/verification action on unencrypted HTTP link.\')\n    elif action and score >= 33:\n        score = max(score, 53)\n        reasons.append(\'Account/verification action adds risk to an already unusual hostname.\')\n    if bait and score >= 33:\n        score = max(score, 58)\n        reasons.append(\'Prize/reward keywords appear with other URL warning signs.\')\n    if port not in (None, 80, 443):\n        score = max(score, 34)\n        reasons.append(\'URL uses a nonstandard network port.\')\n    if len(raw) > 250:\n        score = max(score, 33)\n        reasons.append(\'Unusually long link, inspect the complete destination.\')\n\n    if not reasons:\n        reasons.append(\'No strong offline structural warning found; this does NOT verify the website is safe.\')\n    status = \'high_structural_risk\' if score >= 75 else (\'caution\' if score >= 45 else \'low_indicators_not_verified\')\n    return URLAssessment(min(100, int(score)), tuple(reasons), ascii_host, status, high)\n',encoding='utf-8')
(ROOT/'scripts'/'step49_external_url_evaluation.py').write_text('"""Step49: SOURCE-CLAIMED phishing and popular-domain proxy checks.\n\nCompares local URL structural warnings against two documented *different*\nsources. NEVER opens candidate pages, resolves domains, or fetches redirect\nchains. Network mode downloads ONLY two hard-coded publisher datasets, NOT\nany URL contained in them. No model/threshold selection against these data.\n\nThis is independent-source *diagnostics*, NOT a validated phishing classifier.\nThe Tranco set is a popularity proxy, not a collection of authenticated\nlegitimate webpages. Phishing URLs can occur on trusted hostnames.\n\nOffline usage:\n python scripts/step49_external_url_evaluation.py \\\\\n  --phishing-file path/to/OpenPhish_feed.txt \\\\\n  --proxy-file path/to/Tranco_list.csv --output /private/results\n\nDownload mode (safe publisher feeds only):\n python scripts/step49_external_url_evaluation.py --download --output /private/results\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport random\nimport sys\nimport urllib.request\nimport zipfile\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom phishguard.url_policy import assess_url\n\n# Both values derived from public publishers; do not accept arbitrary fetch URLs.\nOPENPHISH_SOURCE = \'https://raw.githubusercontent.com/openphish/public_feed/refs/heads/main/feed.txt\'\nTRANCO_ID = \'V354N\'  # list created 2026-11-01 (see https://tranco-list.eu/list/V354N)\nTRANCO_SOURCE = f\'https://tranco-list.eu/download_daily/{TRANCO_ID}\'\nSOURCES = {\'openphish\': OPENPHISH_SOURCE, \'tranco\': TRANCO_SOURCE}\nACCEPTED_TYPES = (\'.txt\', \'.csv\', \'.zip\')\nMAX_BYTES = 38 * 1024 * 1024\nN_LIMIT = 400\n\n\ndef _sha(value: str) -> str:\n    return hashlib.sha256(value.encode(\'utf-8\')).hexdigest()\n\n\ndef _download_only_publisher(which: str) -> bytes:\n    """Fixed URL only. NEVER follows a URL *from* either publisher feed."""\n    url = SOURCES[which]\n    req = urllib.request.Request(url, headers={\n        \'User-Agent\': \'PhishGuard-PK-academic-offline-evaluation/1.0\',\n        \'Accept\': \'text/plain,text/csv,application/zip,application/octet-stream\',\n    })\n    # Publisher URL fixed; Python may follow publisher HTTP redirects.\n    # Never pass any malicious/test URL to urlopen.\n    class SameHostOnlyRedirect(urllib.request.HTTPRedirectHandler):\n        def redirect_request(self, req, fp, code, msg, headers, newurl):\n            # Keep requests on the selected publisher host; never follow an\n            # arbitrary malicious redirect when downloading threat feeds.\n            newparts=urlsplit(newurl)\n            if newparts.scheme!=\'https\' or newparts.hostname!=urlsplit(url).hostname:\n                raise ValueError(\'Publisher attempted a cross-host / non-HTTPS redirect\')\n            return super().redirect_request(req,fp,code,msg,headers,newurl)\n    opener=urllib.request.build_opener(SameHostOnlyRedirect())\n    with opener.open(req, timeout=35) as res:\n        data = res.read(MAX_BYTES + 1)\n    if len(data) > MAX_BYTES:\n        raise ValueError(\'Publisher feed exceeds size cap\')\n    return data\n\n\ndef _parse_urls_from_openphish(raw: bytes) -> list[str]:\n    contents = raw.decode(\'utf-8-sig\', errors=\'replace\')\n    seen = set()\n    out = []\n    for line in contents.splitlines():\n        u = line.strip()\n        if not u or u.startswith(\'#\') or u in seen or len(u) > 10000:\n            continue\n        if urlsplit(u).scheme.lower() not in (\'http\',\'https\'):\n            continue\n        if not urlsplit(u).hostname:\n            continue\n        seen.add(u)\n        out.append(u)\n    return out\n\n\ndef _parse_tranco(raw: bytes) -> list[tuple[int, str]]:\n    if raw[:4] == b\'PK\\x03\\x04\':\n        with zipfile.ZipFile(io.BytesIO(raw)) as z:\n            valid = [i for i in z.infolist() if i.filename.lower().endswith(\'.csv\') and not i.is_dir()]\n            if not valid or len(valid) != 1 or valid[0].file_size > MAX_BYTES * 4:\n                raise ValueError(\'Invalid Tranco ZIP structure\')\n            with z.open(valid[0]) as f:\n                content = f.read(MAX_BYTES * 4 + 1)\n                if len(content) > MAX_BYTES * 4:\n                    raise ValueError(\'Tranco uncompressed size cap\')\n    else:\n        content = raw\n    result = []\n    for row in csv.reader(io.StringIO(content.decode(\'utf-8-sig\', errors=\'replace\'))):\n        if len(row) < 2:\n            continue\n        try:\n            rank = int(row[0].strip())\n        except ValueError:\n            continue\n        domain = row[1].strip().lower().strip(\'.\')\n        if not domain or \'/\' in domain or \':\' in domain or \'@\' in domain or \' \' in domain:\n            continue\n        try:\n            domain.encode(\'idna\')\n        except UnicodeError:\n            continue\n        # Tranco entries are base domains; their homepage is only a *proxy*.\n        result.append((rank,domain))\n    return result\n\n\ndef _sample_phishing(urls: list[str], n: int) -> list[str]:\n    # Uniform sample by hostname, one example per hostname; avoid 200 URLs\n    # from the same compromised domain counting as 200 independent successes.\n    by_host: dict[str,str] = {}\n    for url in urls:\n        hostname = (urlsplit(url).hostname or \'\').lower()\n        if hostname not in by_host:\n            by_host[hostname] = url\n    hosts = sorted(by_host)\n    random.Random(20261102).shuffle(hosts)\n    return [by_host[h] for h in hosts[:n]]\n\n\ndef _sample_tranco(rows: list[tuple[int, str]], n: int) -> list[tuple[int,str]]:\n    # Draw a reproducible subset of the *top 100k* domains. This is not a\n    # verified non-phishing set and contains only base-domain homepages.\n    seen = set()\n    eligible=[]\n    for rank,domain in rows:\n        if rank > 100000 or domain in seen:\n            continue\n        seen.add(domain)\n        eligible.append((rank,domain))\n    eligible.sort()\n    random.Random(4902).shuffle(eligible)\n    return eligible[:n]\n\n\ndef _score_cases(urls: list[str], kind: str, rank_map: dict[str,int]|None=None):\n    out = []\n    for url in urls:\n        result=assess_url(url)\n        # Do not include real malicious URLs, full domains, paths, screenshots,\n        # query strings, passwords, or token values in any result artifacts.\n        host = (urlsplit(url).hostname or \'\').lower()\n        row={\'record_hash\':_sha(url), \'host_hash\':_sha(host), \'cohort\':kind,\n             \'score\':result.indicator, \'status\':result.status,\n             \'warning_45\':result.indicator>=45, \'warning_75\':result.indicator>=75,\n             \'rank\':(rank_map or {}).get(host,\'\'),\n             \'signals\':len(result.reasons)}\n        out.append(row)\n    return out\n\n\ndef _metrics(rows:list[dict], positive:bool):\n    n=len(rows)\n    return {\'n\':n, \'warning_45_n\':sum(x[\'warning_45\'] for x in rows),\n            \'warning_75_n\':sum(x[\'warning_75\'] for x in rows),\n            \'warning_45_rate\':round(sum(x[\'warning_45\'] for x in rows)/n,4) if n else None,\n            \'warning_75_rate\':round(sum(x[\'warning_75\'] for x in rows)/n,4) if n else None,\n            \'interpretation\':(\'phishing feed catch rate, NOT verified universal recall\' if positive\n                 else \'warning frequency on popular-domain homepages, NOT false-positive rate on authenticated safe sites\')}\n\n\ndef evaluate(openphish_raw:bytes|None, tranco_raw:bytes|None, output:Path,\n             source_mode:str, n:int=N_LIMIT, errors:dict[str,str]|None=None) -> dict:\n    output.mkdir(parents=True,exist_ok=True)\n    report={\'experiment\':\'STEP49_FRESH_SOURCE_URL_DIAGNOSTIC_NOT_FINAL_MODEL_EVALUATION\',\n            \'timestamp_utc\':datetime.now(timezone.utc).isoformat(timespec=\'seconds\'),\n            \'mode\':source_mode, \'tranco_list_id\':TRANCO_ID,\n            \'sources\':SOURCES, \'fixed_thresholds_not_tuned_on_test\':[45,75],\n            \'record_selection\':\'deterministic host-diverse OpenPhish sample, deterministic top-100k Tranco domain homepage sample\',\n            \'any_candidate_URL_opened\':False,\n            \'safety\':\'No candidate URL requests, DNS resolution or content fetch. Only publishers fetched when --download set.\',\n            \'source_and_label_limits\':[\n              \'OpenPhish is provider-labelled feed, not a separately human-authenticated Pakistani phishing dataset.\',\n              \'Tranco rank is not proof of domain safety, and homepage proxies underrepresent legitimate deep paths.\',\n              \'Matching host names between two sources need investigation; do not infer ground truth from either alone.\',\n              \'This is not a random representative sample of 2026 Pakistani SMS phishing or current web traffic.\',\n              \'Not a calibration study, no confidence intervals or F1/accuracy claims, no trained model deployment allowed.\'\n            ],\n            \'url_model_deployment_approved\':False,\n            \'independent_50_100_real_SMS_evaluation_done\':False,\n            \'publisher_errors\':errors or {}}\n    rows=[]\n    host_sets={}\n    for name, raw in [(\'openphish\',openphish_raw),(\'tranco\',tranco_raw)]:\n        if raw is None:\n            report[name]={\'status\':\'NOT_AVAILABLE\',\'n\':0}\n            continue\n        report[f\'{name}_publisher_bytes_sha256\']=hashlib.sha256(raw).hexdigest()\n        if name==\'openphish\':\n            candidates=_parse_urls_from_openphish(raw)\n            chosen=_sample_phishing(candidates,n)\n            curr=_score_cases(chosen,\'OpenPhish_feed_label\')\n            host_sets[name]={urlsplit(x).hostname.lower() for x in chosen}\n            report[name]={\'status\':\'SOURCE_LABELLED_NOT_INDEPENDENTLY_REVIEWED\',\'raw_valid_unique_urls\':len(candidates),\n                          \'selected_distinct_hosts\':len(chosen),**_metrics(curr,positive=True)}\n        else:\n            candidates=_parse_tranco(raw)\n            chosen=_sample_tranco(candidates,n)\n            rank_map={d:i for i,d in chosen}\n            curr=_score_cases([\'https://\'+domain+\'/\' for i,domain in chosen],\'Tranco_top100k_homepage_proxy\',rank_map)\n            host_sets[name]={d for _,d in chosen}\n            report[name]={\'status\':\'POPULAR_DOMAIN_PROXY_NOT_VERIFIED_BENIGN\',\'raw_parsed_domains\':len(candidates),\n                          \'selected_distinct_hosts\':len(chosen),**_metrics(curr,positive=False)}\n        rows+=curr\n    if len(host_sets)==2:\n        report[\'selected_hostname_overlap_n\']=len(host_sets[\'openphish\'] & host_sets[\'tranco\'])\n    if rows:\n        path=output/\'STEP49_HASHED_CASES_ONLY.csv\'\n        with path.open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n            w=csv.DictWriter(f,fieldnames=list(rows[0]))\n            w.writeheader();w.writerows(rows)\n    report[\'result_status\']=\'INCOMPLETE_SOURCE_FETCH\' if len(host_sets)<2 else \'RESEARCH_DIAGNOSTIC_COMPLETE_NO_DEPLOYMENT\'\n    (output/\'STEP49_SOURCE_DIAGNOSTIC.json\').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    return report\n\n\ndef main(args=None):\n    p=argparse.ArgumentParser()\n    g=p.add_mutually_exclusive_group(required=True)\n    g.add_argument(\'--download\', action=\'store_true\',help=\'download only two fixed publisher feeds\')\n    g.add_argument(\'--phishing-file\',type=Path,help=\'downloaded public OpenPhish feed locally\')\n    p.add_argument(\'--proxy-file\',type=Path,help=\'local Tranco CSV or ZIP; required with --phishing-file\')\n    p.add_argument(\'--output\',type=Path,default=Path.cwd()/\'step49_private_output\')\n    p.add_argument(\'--n\',type=int,default=N_LIMIT)\n    a=p.parse_args(args)\n    if not 1<=a.n<=2000:\n        p.error(\'--n must be 1..2000\')\n    if a.phishing_file and not a.proxy_file:\n        p.error(\'--proxy-file required with --phishing-file\')\n    raw={};errors={}\n    if a.download:\n        for which in SOURCES:\n            try:\n                raw[which]=_download_only_publisher(which)\n            except Exception as exc:\n                errors[which]=f\'{type(exc).__name__}: {str(exc)[:200]}\'\n                raw[which]=None\n    else:\n        raw={\'openphish\':a.phishing_file.read_bytes(),\'tranco\':a.proxy_file.read_bytes()}\n    report=evaluate(raw[\'openphish\'],raw[\'tranco\'],a.output,\n                    \'LIVE_PUBLISHER_FEEDS\' if a.download else \'SUPPLIED_PUBLISHER_FILES\',a.n,errors)\n    print(json.dumps({key:report[key] for key in [\'result_status\',\'openphish\',\'tranco\',\'publisher_errors\',\'url_model_deployment_approved\']},indent=2))\n    return report\n\nif __name__==\'__main__\':\n    main()\n',encoding='utf-8')
(ROOT/'scripts'/'step50_supervised_external_research.py').write_text('"""Step 50: Dataset-separated phishing URL model research, NEVER deploy automatically.\n\nInputs: (a) 10k UCI PhiUSIIL 2024 URLs, labels 0=phishing 1=legitimate,\n(b) fresh OpenPhish provider-labelled URLs, (c) dated Tranco domain homepage proxies.\nOnly passive URL string processing. No URLs in the datasets are requested, visited,\nresolved, or dereferenced; only allowlisted publisher download operations happen\noutside this file in the one-click Colab notebook.\n\nExternal feed is OUT-OF-DOMAIN research evidence, not authenticated ground truth.\nAll external records in output are SHA-256 hashed. Exported model always named\nRESEARCH_ONLY, never models/url_model_real.joblib.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ctypes\nimport ctypes.util\nimport hashlib\nimport json\nimport os\nimport re\nfrom collections import Counter\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import SGDClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, recall_score\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom sklearn.pipeline import Pipeline\n\nSAFE_EXAMPLES = [\n \'https://github.com/\',\'https://github.com/openai\',\'https://en.wikipedia.org/wiki/Artificial_intelligence\',\n \'https://www.wikipedia.org\',\'https://www.python.org\',\'https://docs.python.org/3/\',\n \'https://www.microsoft.com\',\'https://support.microsoft.com/\',\'https://developer.mozilla.org/en-US/\',\n \'https://www.cloudflare.com\',\'https://pypi.org/project/fastapi/\',\'https://www.nasa.gov\',\n \'https://www.unicef.org\',\'https://www.who.int\',\'https://www.un.org\',\'https://www.openai.com\',\n \'https://www.bbc.com\',\'https://www.google.com\',\'https://docs.google.com\',\'https://www.apple.com\',\n \'https://www.amazon.com\',\'https://www.khanacademy.org\',\'https://www.ibm.com\',\n \'https://www.sbp.org.pk\',\'https://www.fbr.gov.pk\',\'https://www.irs.gov\',\'https://www.gov.uk\',\n \'https://example.org/\',\'https://www.youtube.com/\',\'https://www.reddit.com/\'\n]\n\n_psl_lib = None\n_psl_context = None\ntry:\n    _psl_lib = ctypes.CDLL(ctypes.util.find_library(\'psl\') or \'libpsl.so.5\')\n    _psl_lib.psl_builtin.restype=ctypes.c_void_p\n    _psl_lib.psl_registrable_domain.argtypes=[ctypes.c_void_p,ctypes.c_char_p]\n    _psl_lib.psl_registrable_domain.restype=ctypes.c_char_p\n    _psl_context=_psl_lib.psl_builtin()\nexcept Exception:\n    _psl_lib=None\n\n\ndef host_of(url:str)->str:\n    try:\n        p=urlsplit(url if re.match(r\'^https?://\',url,re.I) else \'http://\'+url)\n        h=(p.hostname or \'\').strip(\'.\').lower().encode(\'idna\').decode(\'ascii\')\n        return h if h and len(h)<254 and not any(c.isspace() for c in h) else \'\'\n    except (ValueError,UnicodeError,TypeError):\n        return \'\'\n\n\ndef registered_group(url:str)->str:\n    host=host_of(url)\n    if not host:return \'\'\n    if _psl_lib is not None:\n        try:\n            res=_psl_lib.psl_registrable_domain(_psl_context,host.encode(\'ascii\'))\n            if res:return res.decode(\'ascii\')\n        except Exception:pass\n    # Conservative fallback: exact hostname only; disclose limitation.\n    return host\n\n\ndef site_group_quality()->str:\n    return \'libpsl_registrable_domain\' if _psl_lib is not None else \'EXACT_HOST_FALLBACK_NOT_REGISTRABLE\'\n\n\ndef lexical_features(url:str)->list[float]:\n    p=urlsplit(url if re.match(r\'^https?://\',url,re.I) else \'http://\'+url)\n    h=host_of(url); z=url.lower()\n    digits=sum(ch.isdigit() for ch in z)\n    return [min(len(z),3000),len(h),z.count(\'.\'),z.count(\'-\'),z.count(\'@\'),digits,z.count(\'/\'),\n            z.count(\'?\'),z.count(\'=\'),z.count(\'%\'),z.count(\'&\'),z.count(\'#\'),\n            len(p.path),len(p.query),len(h.split(\'.\')),int(p.scheme==\'https\'),\n            int(\'xn--\' in h),int(bool(re.search(r\'\\b(?:verify|claim|login|bonus|reward|kyc|otp|prize)\\b\',z))),\n            int(bool(re.search(r\'\\b(?:account|wallet|payment|update|password|security)\\b\',z)))]\n\n\ndef read_uci(path:Path)->tuple[pd.DataFrame,dict]:\n    raw=pd.read_csv(path,dtype=str,keep_default_na=False)\n    assert {\'url\',\'label_uci\'}.issubset(raw.columns),\'Expected URL and source-label columns\'\n    x=raw[[\'url\',\'label_uci\']].copy();x.url=x.url.str.strip()\n    assert set(x.label_uci.unique())<= {\'0\',\'1\'},\'Unknown UCI label values\'\n    x[\'label\']=1-x.label_uci.astype(int) # model: 1 phishing, 0 legitimate\n    x=x[x.url.str.len().between(5,2000)].copy()\n    # Contradictory labels for exact same URL are excluded, not arbitrarily resolved.\n    conflicts=x.groupby(\'url\').label.nunique();bad=set(conflicts[conflicts>1].index)\n    x=x[~x.url.isin(bad)].drop_duplicates(\'url\').copy()\n    x[\'group\']=x.url.map(registered_group)\n    x=x[x.group.ne(\'\')].reset_index(drop=True)\n    return x,{\'input_rows\':int(len(raw)), \'usable_after_dedup\':int(len(x)),\n              \'conflicting_url_values_removed\':int(len(bad)),\'grouping_quality\':site_group_quality()}\n\n\ndef make_splits(df:pd.DataFrame):\n    for seed in (2026,42,17,129,4902,6301,78):\n        g=GroupShuffleSplit(n_splits=1,train_size=.70,random_state=seed)\n        a,b=next(g.split(df,df.label,df.group));tr=df.iloc[a].copy();rest=df.iloc[b].copy()\n        v,t=next(GroupShuffleSplit(n_splits=1,train_size=.5,random_state=seed+1).split(rest,rest.label,rest.group))\n        val=rest.iloc[v].copy();te=rest.iloc[t].copy()\n        if all(z.label.nunique()==2 and len(z)>=100 for z in (tr,val,te)):\n            return tr,val,te\n    raise ValueError(\'Unable to form viable group-disjoint splits\')\n\n\ndef metrics(y,pr)->dict:\n    p,r,f,_=precision_recall_fscore_support(y,pr,labels=[0,1],average=\'macro\',zero_division=0)\n    cm=confusion_matrix(y,pr,labels=[0,1]).tolist()\n    return {\'n\':int(len(y)), \'accuracy\':round(float(accuracy_score(y,pr)),4),\n            \'precision_macro\':round(float(p),4),\'recall_macro\':round(float(r),4),\n            \'f1_macro\':round(float(f),4),\'phishing_recall\':round(float(recall_score(y,pr,pos_label=1,zero_division=0)),4),\n            \'false_positive_rate\':round(float(cm[0][1]/max(1,sum(cm[0]))),4),\n            \'confusion_matrix_labels_0_benign_1_phishing\':cm}\n\n\ndef pick_threshold(y,probas)->tuple[float,dict]:\n    # ALL threshold selection restricted to UCI validation; external sources never tune it.\n    options=[]\n    for threshold in np.arange(.25,.951,.05):\n        t=round(float(threshold),2);m=metrics(y,probas>=t)\n        options.append((m[\'f1_macro\'],m[\'phishing_recall\'],t,m))\n    eligible=[x for x in options if x[3][\'false_positive_rate\']<=.05]\n    if not eligible:\n        return 0.5,{\'reason\':\'No validation threshold achieved FPR<=5%; default 0.5 used\',\'constraints_met\':False}\n    _,_,t,m=max(eligible,key=lambda x:(x[0],x[1]))\n    return t,{\'reason\':\'Validation-only macro F1 under FPR<=5% constraint\',\'constraints_met\':True,\'selected_validation\':m}\n\n\ndef _csv_hash_records(urls,probs,cohort,threshold):\n    records=[]\n    for url,pr in zip(urls,probs):\n        records.append({\'url_sha256\':hashlib.sha256(url.encode()).hexdigest(),\n          \'hostname_sha256\':hashlib.sha256(host_of(url).encode()).hexdigest(),\n          \'cohort\':cohort,\'model_score_not_calibrated\':round(float(pr),4),\n          \'warning\':bool(pr>=threshold)})\n    return records\n\n\ndef train_and_evaluate(uci_file:Path,output:Path,openphish:list[str]|None=None,\n                       tranco_domains:list[str]|None=None, baseline=None)->dict:\n    output.mkdir(parents=True,exist_ok=True)\n    d, audit=read_uci(uci_file)\n    ext_ph=list(dict.fromkeys(openphish or []))\n    ext_tr=list(dict.fromkeys(tranco_domains or []))\n    # External samples and reference examples held out of training by registered group.\n    ph_urls=[u for u in ext_ph if host_of(u)]\n    tr_urls=[\'https://\'+h.strip(\'/\')+\'/\' for h in ext_tr if h and host_of(\'https://\'+h)]\n    ext_groups={registered_group(u) for u in ph_urls+tr_urls+SAFE_EXAMPLES}\n    ext_groups.discard(\'\')\n    before=len(d)\n    d=d[~d.group.isin(ext_groups)].copy()\n    audit[\'excluded_external_or_reference_groups_rows\']=int(before-len(d))\n    assert len(d)>=1000 and d.label.nunique()==2, \'Insufficient data after source separation\'\n    train,val,test=make_splits(d)\n    groups=[set(x.group) for x in (train,val,test)]\n    assert not(groups[0]&groups[1] or groups[0]&groups[2] or groups[1]&groups[2])\n    assert not((groups[0]|groups[1]|groups[2])&ext_groups)\n    cv=Pipeline([(\'tfidf\',TfidfVectorizer(analyzer=\'char\',ngram_range=(3,5),min_df=2,max_features=65000,sublinear_tf=True)),\n       (\'classifier\',SGDClassifier(loss=\'log_loss\',alpha=2e-5,max_iter=1200,tol=1e-3,class_weight=\'balanced\',random_state=2026))])\n    cv.fit(train.url.astype(str).tolist(),train.label.to_numpy())\n    cv_val=cv.predict_proba(val.url.astype(str).tolist())[:,list(cv.classes_).index(1)]\n    lex=RandomForestClassifier(n_estimators=120,max_depth=15,min_samples_leaf=2,class_weight=\'balanced\',n_jobs=-1,random_state=2026)\n    xtr=np.asarray([lexical_features(u) for u in train.url],np.float32)\n    xva=np.asarray([lexical_features(u) for u in val.url],np.float32)\n    lex.fit(xtr,train.label.to_numpy())\n    lex_val=lex.predict_proba(xva)[:,list(lex.classes_).index(1)]\n    candidates=[]\n    for name,model,vals in [(\'char_tfidf_linear\',cv,cv_val),(\'lexical_random_forest\',lex,lex_val)]:\n        thresh,details=pick_threshold(val.label.to_numpy(),vals)\n        m=metrics(val.label.to_numpy(),vals>=thresh)\n        candidates.append({\'name\':name,\'model\':model,\'threshold\':thresh,\n                           \'val\':m,\'threshold_selection\':details})\n    winner=max(candidates,key=lambda x:(x[\'threshold_selection\'][\'constraints_met\'],x[\'val\'][\'f1_macro\']))\n    def probability(model,urls):\n        if model is cv:return model.predict_proba(list(urls))[:,list(model.classes_).index(1)]\n        return model.predict_proba(np.asarray([lexical_features(u) for u in urls],np.float32))[:,list(model.classes_).index(1)]\n    th=winner[\'threshold\'];mdl=winner[\'model\']\n    ytest=test.label.to_numpy();p=probability(mdl,test.url.tolist());testm=metrics(ytest,p>=th)\n    safe_proba=probability(mdl,SAFE_EXAMPLES)\n    ext={}\n    hashed=[]\n    for name,urls in [(\'openphish_provider_label\',ph_urls),(\'tranco_homepage_proxy\',tr_urls)]:\n        if not urls:\n            ext[name]={\'n\':0,\'status\':\'SOURCE_UNAVAILABLE_NOT_EVALUATED\'};continue\n        probs=probability(mdl,urls)\n        ext[name]={\'n\':len(urls),\'warnings_n\':int(sum(probs>=th)),\n                   \'warnings_fraction\':round(float(np.mean(probs>=th)),4),\n                   \'mean_uncalibrated_score\':round(float(np.mean(probs)),4),\n                   \'status\':\'SOURCE_LABELLED_NOT_INDEPENDENTLY_VERIFIED\' if name.startswith(\'openphish\') else \'POPULAR_HOMEPAGE_PROXY_NOT_VERIFIED_SAFE\'}\n        hashed+=_csv_hash_records(urls,probs,name,th)\n    if baseline:\n        for name,urls in [(\'openphish_provider_label\',ph_urls),(\'tranco_homepage_proxy\',tr_urls)]:\n            if urls:ext[name][\'baseline_structural_45_warnings_n\']=int(sum(baseline(u).indicator>=45 for u in urls))\n    # Guard: predeclared diagnostic limits; still never auto-deploy even if pass.\n    ph=ext.get(\'openphish_provider_label\',{});leg=ext.get(\'tranco_homepage_proxy\',{})\n    stress=int(sum(safe_proba>=th))\n    diagnostic_pass=bool(ph.get(\'n\',0)>=100 and leg.get(\'n\',0)>=100 and\n           ph.get(\'warnings_fraction\',0)>=.80 and leg.get(\'warnings_fraction\',1)<=.05 and stress<=1 and\n           testm[\'phishing_recall\']>=.90 and testm[\'false_positive_rate\']<=.05 and\n           winner[\'threshold_selection\'][\'constraints_met\'] and site_group_quality()==\'libpsl_registrable_domain\')\n    summary={\'experiment\':\'STEP50_UCI_2024_TRAIN_FRESH_FEEDS_SEPARATE_RESEARCH\',\n      \'data\':audit, \'labels\':\'UCI source: 1 legitimate, 0 phishing; internal: 1 phishing, 0 legitimate\',\n      \'split_rows\':{k:int(len(v)) for k,v in [(\'train\',train),(\'validation\',val),(\'test\',test)]},\n      \'split_registered_domain_overlap\':0,\'external_group_overlap_with_training\':0,\n      \'model_candidates_validation\':[{k:v for k,v in a.items() if k!=\'model\'} for a in candidates],\n      \'winner\':winner[\'name\'],\'threshold_selected_on_validation\':th,\n      \'uci_heldout_source_label_metrics\':testm,\n      \'external_source_diagnostics\':ext,\n      \'legacy_30_reference_urls_warning_count\':stress,\n      \'diagnostic_gate_pass\':diagnostic_pass,\n      \'deployment_approved\':False,\n      \'deployment_note\':\'NO DEPLOYMENT. Diagnostic gate alone is insufficient. Needs independent real-world and Pakistani URL validation.\',\n      \'safety\':\'Only URL strings classified; no candidate URLs opened, resolved, or requested.\',\n      \'risks\':[\'UCI 2024 aged data and possible template/source artifacts\',\n               \'Provider-labelled OpenPhish feed not individually verified\',\n               \'Tranco popular domains are NOT authenticated legitimate URLs\',\n               \'Homepage proxies omit legitimate deep links\',\n               \'Scores are NOT calibrated probabilities or guarantees of safety\']}\n    (output/\'STEP50_RESULTS.json\').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding=\'utf8\')\n    pd.DataFrame(hashed,columns=[\'url_sha256\',\'hostname_sha256\',\'cohort\',\'model_score_not_calibrated\',\'warning\']).to_csv(output/\'STEP50_EXTERNAL_HASHED_ONLY.csv\',index=False)\n    (output/\'DO_NOT_DEPLOY.txt\').write_text(\'RESEARCH ONLY: UCI 2024 labels and 2026 provider/proxy diagnostics are not independent real-world accuracy. NEVER upload model to live site.\\n\')\n    joblib.dump({\'model\':mdl,\'threshold\':th,\'model_type\':winner[\'name\'], \'label_mapping\':{\'0\':\'legitimate\',\'1\':\'phishing\'},\n                 \'deployment_approved\':False},output/\'STEP50_MODEL_RESEARCH_ONLY.joblib\',compress=3)\n    return summary\n\nif __name__==\'__main__\':\n    a=argparse.ArgumentParser()\n    a.add_argument(\'--uci-file\',type=Path,required=True)\n    a.add_argument(\'--output\',type=Path,required=True)\n    a.add_argument(\'--openphish-file\',type=Path)\n    a.add_argument(\'--tranco-file\',type=Path)\n    args=a.parse_args()\n    op=args.openphish_file.read_text().splitlines() if args.openphish_file else None\n    td=args.tranco_file.read_text().splitlines() if args.tranco_file else None\n    report=train_and_evaluate(args.uci_file,args.output,op,td)\n    print(json.dumps({k:report[k] for k in (\'winner\',\'threshold_selected_on_validation\',\'uci_heldout_source_label_metrics\',\'external_source_diagnostics\',\'legacy_30_reference_urls_warning_count\',\'deployment_approved\')},indent=2))\n',encoding='utf-8')
(RootCSV:=ROOT/'input'/'UCI_10K_PUBLIC_SOURCE_LABELED_NOT_VERIFIED.csv').write_bytes(gzip.decompress(base64.b64decode('...')))
import sys
sys.path.insert(0,str(ROOT))
from scripts.step50_supervised_external_research import read_uci,train_and_evaluate
from scripts.step49_external_url_evaluation import (_download_only_publisher,_parse_urls_from_openphish,_parse_tranco,_sample_phishing,_sample_tranco,TRANCO_ID,OPENPHISH_SOURCE,TRANCO_SOURCE)
from phishguard.url_policy import assess_url
import sklearn,pandas
print('Source records:',len(read_uci(RootCSV)[0]),' | sklearn:',sklearn.__version__,' | pandas:',pandas.__version__)
print('Only public publisher files will be downloaded. No suspicious URL will be opened.')


In [ ]:
import hashlib
publisher_errors={}
feed_urls=[]
rank_domains=[]
publisher_audit={}
try:
 raw=_download_only_publisher('openphish')
 candidate=_parse_urls_from_openphish(raw)
 feed_urls=_sample_phishing(candidate,400)
 publisher_audit['openphish']={'source':OPENPHISH_SOURCE,'raw_unique_urls':len(candidate),'selected_distinct_hosts':len(feed_urls),'source_sha256':hashlib.sha256(raw).hexdigest()}
 print('OpenPhish: selected',len(feed_urls),'unique host URLs (provider labels)')
except Exception as ex:
 publisher_errors['openphish']=f'{type(ex).__name__}: {str(ex)[:250]}'
 print('OpenPhish unavailable:',publisher_errors['openphish'])
try:
 raw=_download_only_publisher('tranco')
 candidate=_parse_tranco(raw)
 picked=_sample_tranco(candidate,400)
 rank_domains=[domain for _,domain in picked]
 publisher_audit['tranco']={'source':TRANCO_SOURCE,'list_id':TRANCO_ID,'raw_domains':len(candidate),'selected_distinct_hosts':len(rank_domains),'source_sha256':hashlib.sha256(raw).hexdigest()}
 print('Tranco: selected',len(rank_domains),'top-ranked domain homepage proxies')
except Exception as ex:
 publisher_errors['tranco']=f'{type(ex).__name__}: {str(ex)[:250]}'
 print('Tranco unavailable:',publisher_errors['tranco'])


In [ ]:
import json,sys
from datetime import datetime,timezone
try:
 report=train_and_evaluate(RootCSV,ROOT/'results',feed_urls,rank_domains,baseline=assess_url)
 report['publisher_errors']=publisher_errors
 report['publisher_evidence']=publisher_audit
 report['runtime_utc']=datetime.now(timezone.utc).isoformat()
 (ROOT/'results'/'STEP50_RESULTS.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
 print('Model:',report['winner'],'| UCI holdout:',report['uci_heldout_source_label_metrics'])
 print('External:',report['external_source_diagnostics'])
 print('30 safe reference warnings:',report['legacy_30_reference_urls_warning_count'])
 print('Deployment approved:',report['deployment_approved'])
except Exception as e:
 err={'error':f'{type(e).__name__}: {str(e)}','stage':'STEP50_TRAINING_FAILED','publisher_errors':publisher_errors,'deployment_approved':False}
 (ROOT/'results'/'STEP50_FAILURE_DIAGNOSTIC.json').write_text(json.dumps(err,indent=2),encoding='utf8')
 print('Training failed, see diagnostic:',err)


In [ ]:
import zipfile
from google.colab import files
p=Path('/content/PhishGuard_Step50_External_Model_Results.zip')
with zipfile.ZipFile(p,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for f in sorted((ROOT/'results').glob('*')):
  if f.is_file(): z.write(f,f.name)
print('Results ZIP created:',p.name,'bytes',p.stat().st_size)
print('This archive contains NO raw phishing URLs. Do NOT deploy the model.')
files.download(str(p))
